# Análise VarejoMix — vendas omnichannel e features de clientes

Notebook reproduzível. Se `DATABASE_URL` aponta para a instância carregada, a extração usa PostgreSQL/SQLAlchemy e `pandas.read_sql`. Sem conexão configurada, o notebook demonstra a consulta com SQLite em memória sobre o Gold local. Os eventos deste exemplo vêm das fixtures JSON. A data de referência (2025-04-01) fecha as features; não se treina modelo.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd
from sqlalchemy import create_engine, text
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path(".." ).resolve()
sys.path.insert(0, str(ROOT / "python"))
from build_features import build_features


In [ ]:
database_url = os.getenv("DATABASE_URL")
if database_url:
    engine = create_engine(database_url, pool_pre_ping=True)
    monthly_sales = pd.read_sql(text("SELECT * FROM dw.vw_monthly_channel_sales ORDER BY year_month, channel_name"), engine)
    customers = pd.read_sql(text("SELECT customer_id FROM customers ORDER BY customer_id"), engine)
    fact = pd.read_sql(text("""SELECT f.order_item_id, f.order_id, dc.customer_id, d.calendar_date AS order_date,
        ch.channel_name AS channel, f.quantity, f.returned_quantity, f.net_amount
        FROM dw.fact_sales f JOIN dw.dim_customer dc USING(customer_key)
        JOIN dw.dim_date d USING(date_key) JOIN dw.dim_channel ch USING(channel_key)
        ORDER BY f.order_item_id"""), engine)
    source_mode = "PostgreSQL via SQLAlchemy/pandas.read_sql"
else:
    # Fallback sem servidor: importa fact parquet para SQLite via SQLAlchemy e consulta com read_sql.
    customers = pd.read_csv(ROOT / "data/source/customers.csv")[["customer_id"]]
    fact = pd.read_parquet(ROOT / "data/gold/fact_sales.parquet")
    engine = create_engine("sqlite://")
    with engine.begin() as connection:
        fact.to_sql("fact_sales", connection, index=False, if_exists="replace")
    monthly_sales = pd.read_sql(text("""SELECT order_month AS year_month, channel, COUNT(DISTINCT order_id) AS order_count,
        SUM(net_amount) AS net_revenue FROM fact_sales GROUP BY order_month, channel ORDER BY order_month, channel"""), engine)
    source_mode = "SQLite/Gold fixture fallback via SQLAlchemy/pandas.read_sql"
print("Origem:", source_mode, "| clientes:", len(customers), "| itens Gold:", len(fact))
monthly_sales.head(12)

Origem: PostgreSQL via SQLAlchemy/pandas.read_sql | clientes: 10 | itens Gold: 21


,year_month,channel_name,order_count,gross_revenue,net_revenue,estimated_margin
0,2025-01,ecommerce,2,67.50,65.50,29.50
1,2025-01,loja,2,1138.89,1078.89,305.89
2,2025-02,ecommerce,3,1256.29,1163.29,321.29
3,2025-02,loja,2,363.00,348.00,178.00
4,2025-03,ecommerce,5,1447.89,1348.89,437.89
5,2025-03,loja,3,397.90,397.90,209.90


## Limpeza e construção das features

A extração de fato preserva a granularidade de linha por item. A agregação a nível de pedido precede o cálculo de ticket médio, de forma que pedidos com mais itens não ganhem peso. O canal predominante é contado por número de pedidos; em empate, usa-se ordem alfabética. Eventos sem cliente continuam disponíveis para o funil, mas não alimentam features individuais.

In [ ]:
events = json.loads((ROOT / "data/source/events.json").read_text(encoding="utf-8"))
reference_date = "2025-04-01"
features = build_features(customers, fact, events, as_of=reference_date)
out = ROOT / "data/gold/customer_features.parquet"
out.parent.mkdir(parents=True, exist_ok=True)
features.to_parquet(out, index=False, engine="pyarrow")
print(f"Dataset: {len(features)} linhas × {len(features.columns)} colunas | corte: {reference_date}")
print("Nulos por coluna:")
display(features.isna().sum().to_frame("nulls"))
display(features.head())

Dataset: 10 linhas × 12 colunas | corte: 2025-04-01
Nulos por coluna:


,nulls
customer_id,0
reference_date,0
last_purchase_date,1
recency_days,1
purchase_frequency,0
monetary_total,0
average_ticket,1
total_items,0
predominant_channel,1
days_since_first_purchase,1


,customer_id,reference_date,last_purchase_date,recency_days,purchase_frequency,monetary_total,average_ticket,total_items,predominant_channel,days_since_first_purchase,event_count,session_count
0,1,2025-04-01,2025-03-30,2.0,4,1191.29,297.82,5,ecommerce,88.0,5,1
1,2,2025-04-01,2025-02-15,45.0,2,242.00,121.00,4,ecommerce,87.0,4,1
2,3,2025-04-01,2025-03-05,27.0,2,194.00,97.00,2,ecommerce,81.0,4,1
3,4,2025-04-01,2025-03-10,22.0,2,352.50,176.25,5,ecommerce,58.0,2,1
4,5,2025-04-01,2025-03-12,20.0,2,234.00,117.00,5,ecommerce,55.0,4,1


## Limitações e interpretação

Recência e frequência descrevem apenas pedidos concluídos observados antes do corte. Cliente sem pedido fica com frequência zero e recência/ticket/canal nulos. A contagem de eventos deriva das fixtures comportamentais disponíveis. Não há rótulo futuro nem treinamento ou validação de modelo; o artefato é um dataset preparado para estudos posteriores, sujeito a viés de cobertura e à pequena escala sintética.

In [ ]:
roundtrip = pd.read_parquet(out)
assert roundtrip.customer_id.is_unique
assert roundtrip.reference_date.eq(reference_date).all()
assert roundtrip.shape == features.shape
print("Parquet gravado e relido:", out, "| unicidade customer_id: OK")
engine.dispose()

Parquet gravado e relido: /home/ubuntu/VarejoMix/data/gold/customer_features.parquet | unicidade customer_id: OK
